# Atom Training on Google Colab

Bootstrap, preflight, quick infra check, full run, and resume in one notebook.



## 1) Mount Drive


In [1]:
from google.colab import drive
#drive.mount('/content/drive')
drive.mount("/content/drive", force_remount=True)


Mounted at /content/drive


## 2) Configure variables

Defaults are set for your current public repo/branch, but values are only set if missing.



In [2]:
import os
import shlex
import subprocess

# Repo defaults (only applied when missing)
os.environ.setdefault('ATOM_REPO_URL', 'https://github.com/MBifolco/atom.git')
os.environ.setdefault('ATOM_BRANCH', 'main')

# Runtime/cache defaults
os.environ.setdefault('ATOM_DRIVE_REPO', '/content/drive/MyDrive/dev/atom')
os.environ.setdefault('ATOM_WORK_REPO', '/content/atom')
os.environ.setdefault('ATOM_INSTALL_JAX_CUDA', '1')
os.environ.setdefault('ATOM_JAX_VERSION', '0.7.2')
os.environ.setdefault('ATOM_DRIVE_REPO_SYNC_MODE', 'stash')  # stash|reset|skip_pull

# Training defaults
os.environ.setdefault('ATOM_USE_VMAP', '1')
os.environ.setdefault('ATOM_SMOKE_OUTPUT_DIR', '/content/drive/MyDrive/atom_runs/quick_test')
os.environ.setdefault('ATOM_FULL_OUTPUT_DIR', '/content/drive/MyDrive/atom_runs/run1')
os.environ.setdefault('ATOM_RESUME_CHECKPOINT_DIR', os.environ['ATOM_FULL_OUTPUT_DIR'])
os.environ.setdefault('ATOM_RESUME_START_GEN', '8')
os.environ.setdefault('ATOM_RESUME_TOTAL_GENS', '20')

# Auth defaults (public repo workflow)
os.environ.setdefault('ATOM_USE_GITHUB_TOKEN', '0')
if os.environ.get('ATOM_USE_GITHUB_TOKEN') != '1':
    os.environ.pop('GITHUB_TOKEN', None)

# Private repo option:
# import getpass
# os.environ['ATOM_USE_GITHUB_TOKEN'] = '1'
# os.environ['GITHUB_TOKEN'] = getpass.getpass('GitHub PAT: ').strip()
os.environ['ATOM_BRANCH'] = 'main'
for key in [
    'ATOM_REPO_URL',
    'ATOM_BRANCH',
    'ATOM_DRIVE_REPO',
    'ATOM_WORK_REPO',
    'ATOM_INSTALL_JAX_CUDA',
    'ATOM_JAX_VERSION',
    'ATOM_DRIVE_REPO_SYNC_MODE',
    'ATOM_USE_VMAP',
    'ATOM_SMOKE_OUTPUT_DIR',
    'ATOM_FULL_OUTPUT_DIR',
    'ATOM_RESUME_CHECKPOINT_DIR',
    'ATOM_RESUME_START_GEN',
    'ATOM_RESUME_TOTAL_GENS',
    'ATOM_USE_GITHUB_TOKEN',
]:
    print(f"{key}={os.environ[key]}")
print('GITHUB_TOKEN set =', 'GITHUB_TOKEN' in os.environ)


def run_streaming(cmd, *, cwd, label, raise_on_error=True, env=None):
    printable = " ".join(shlex.quote(str(part)) for part in cmd)
    print(f"[{label}] $ {printable}")
    proc = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    try:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
    finally:
        rc = proc.wait()

    print(f"\n{label} exit code: {rc}")
    if raise_on_error and rc != 0:
        raise RuntimeError(f"{label} failed with exit code {rc}")
    return rc


print('Shared run_streaming helper ready')



ATOM_REPO_URL=https://github.com/MBifolco/atom.git
ATOM_BRANCH=main
ATOM_DRIVE_REPO=/content/drive/MyDrive/dev/atom
ATOM_WORK_REPO=/content/atom
ATOM_INSTALL_JAX_CUDA=1
ATOM_JAX_VERSION=0.7.2
ATOM_DRIVE_REPO_SYNC_MODE=stash
ATOM_USE_VMAP=1
ATOM_SMOKE_OUTPUT_DIR=/content/drive/MyDrive/atom_runs/quick_test
ATOM_FULL_OUTPUT_DIR=/content/drive/MyDrive/atom_runs/run1
ATOM_RESUME_CHECKPOINT_DIR=/content/drive/MyDrive/atom_runs/run1
ATOM_RESUME_START_GEN=8
ATOM_RESUME_TOTAL_GENS=20
ATOM_USE_GITHUB_TOKEN=0
GITHUB_TOKEN set = False
Shared run_streaming helper ready


## 3) Clone (if needed), preflight, and bootstrap

This handles stale partial clones, validates configuration, and then runs `colab_bootstrap.sh`.

The cell now streams output live so you can see clone, sync, rsync, pip, and JAX install progress as it happens.


In [3]:
import os
import shutil
from pathlib import Path

work_repo = os.environ.get("ATOM_WORK_REPO", "/content/atom")
repo_url = os.environ.get("ATOM_REPO_URL", "")
branch = os.environ.get("ATOM_BRANCH", "main")
auth_repo_url = repo_url
sync_mode = os.environ.get("ATOM_DRIVE_REPO_SYNC_MODE", "<unset>")

print(f"Using WORK_REPO={work_repo}")
print(f"Using BRANCH={branch}")
print(f"ATOM_DRIVE_REPO_SYNC_MODE = {sync_mode}")

if not repo_url:
    raise RuntimeError("ATOM_REPO_URL must be set before first clone.")

if "<org>" in repo_url or "<repo>" in repo_url:
    raise RuntimeError("ATOM_REPO_URL still contains placeholders. Set a real repo URL first.")

if (
    os.environ.get("ATOM_USE_GITHUB_TOKEN", "0") == "1"
    and os.environ.get("GITHUB_TOKEN")
    and repo_url.startswith("https://github.com/")
):
    auth_repo_url = repo_url.replace("https://", f"https://{os.environ['GITHUB_TOKEN']}@", 1)

work_repo_path = Path(work_repo)
if work_repo_path.exists() and not (work_repo_path / ".git").exists():
    print(f"Found stale non-git directory at {work_repo}. Removing it...")
    shutil.rmtree(work_repo)

if not (work_repo_path / ".git").exists():
    run_streaming(
        ["git", "clone", "--branch", branch, "--single-branch", auth_repo_url, work_repo],
        cwd=work_repo_path.parent,
        label="work repo clone",
        raise_on_error=True,
    )

run_streaming(
    ["python", "-u", "-m", "src.atom.training.utils.colab_preflight", "--stage", "bootstrap", "--strict"],
    cwd=work_repo,
    label="bootstrap preflight",
    raise_on_error=True,
)

bootstrap_env = os.environ.copy()
bootstrap_env["ATOM_SKIP_PREFLIGHT"] = "1"
run_streaming(
    ["bash", "colab_bootstrap.sh"],
    cwd=work_repo,
    label="colab bootstrap",
    raise_on_error=True,
    env=bootstrap_env,
)



Using WORK_REPO=/content/atom
Using BRANCH=main
ATOM_DRIVE_REPO_SYNC_MODE = stash
[bootstrap preflight] $ python -u -m src.atom.training.utils.colab_preflight --stage bootstrap --strict
Colab preflight (bootstrap)
[PASS] python-version: 3.12.12
[PASS] drive-sync-mode: stash
[PASS] drive-mounted: Drive parent exists: /content/drive/MyDrive/dev
[PASS] command-git: found at /usr/bin/git
[PASS] command-rsync: found at /usr/bin/rsync
[PASS] repo-branch: main
[PASS] repo-url: Drive cache already initialized; ATOM_REPO_URL is optional for this run.
[PASS] work-repo-parent: /content
Summary: 8 passed, 0 warnings, 0 failures

bootstrap preflight exit code: 0
[colab bootstrap] $ bash colab_bootstrap.sh
Updating Drive repo cache (main)...
From https://github.com/MBifolco/atom
   c56a0fd..7f005d7  main       -> origin/main
 * [new branch]      claude/world-models-exploration-ZgL6j -> origin/claude/world-models-exploration-ZgL6j
 * [new branch]      organize   -> origin/organize
Already on 'main'
Y

0

## 4) Sanity checks


In [4]:
%%bash
set -euo pipefail
cd "${ATOM_WORK_REPO:-/content/atom}"

python - <<'PY'
import torch
from src.atom.training.utils.runtime_platform import detect_runtime_platform
print('torch.cuda.is_available =', torch.cuda.is_available())
print('detected runtime platform =', detect_runtime_platform())
PY

nvidia-smi || true


torch.cuda.is_available = True
detected runtime platform = cuda
Sat Mar 21 03:22:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   35C    P8             16W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |        

## 5) Quick infrastructure smoke test (streaming)

This validates wiring and streams logs live. Quick mode may fail graduation by design.



In [ ]:
import os

work_repo = os.environ.get("ATOM_WORK_REPO", "/content/atom")
smoke_output_dir = os.environ.get("ATOM_SMOKE_OUTPUT_DIR", "/content/drive/MyDrive/atom_runs/quick_test")
use_vmap = os.environ.get("ATOM_USE_VMAP", "1") == "1"

preflight_cmd = [
    "python", "-u", "-m", "src.atom.training.utils.colab_preflight",
    "--stage", "smoke",
    "--output-dir", smoke_output_dir,
    "--strict",
]
if use_vmap:
    preflight_cmd.append("--require-gpu")
run_streaming(preflight_cmd, cwd=work_repo, label="smoke preflight", raise_on_error=True)

train_cmd = [
    "python", "-u", "train_progressive.py",
    "--mode", "quick",
    "--device", "auto",
    "--output-dir", smoke_output_dir,
]
if use_vmap:
    train_cmd.append("--use-vmap")

run_streaming(train_cmd, cwd=work_repo, label="quick smoke", raise_on_error=False)



[smoke preflight] $ python -u -m src.atom.training.utils.colab_preflight --stage smoke --output-dir /content/drive/MyDrive/atom_runs/quick_test --strict --require-gpu
Colab preflight (smoke)
[PASS] python-version: 3.12.12
[PASS] drive-sync-mode: stash
[PASS] drive-mounted: Drive parent exists: /content/drive/MyDrive/dev
[PASS] work-repo: /content/atom
[PASS] train-entrypoint: /content/atom/train_progressive.py
[PASS] python-module-chex: installed
[PASS] output-dir: /content/drive/MyDrive/atom_runs/quick_test
[PASS] gpu-runtime: Detected accelerator platform: cuda
Summary: 8 passed, 0 warnings, 0 failures

smoke preflight exit code: 0
[quick smoke] $ python -u train_progressive.py --mode quick --device auto --output-dir /content/drive/MyDrive/atom_runs/quick_test --use-vmap
2026-03-21 03:22:46.012476: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orde

## 6) Full training run (real run, streaming)



In [ ]:
import os

work_repo = os.environ.get("ATOM_WORK_REPO", "/content/atom")
full_output_dir = os.environ.get("ATOM_FULL_OUTPUT_DIR", "/content/drive/MyDrive/atom_runs/run1")
use_vmap = os.environ.get("ATOM_USE_VMAP", "1") == "1"

preflight_cmd = [
    "python", "-u", "-m", "src.atom.training.utils.colab_preflight",
    "--stage", "full",
    "--output-dir", full_output_dir,
    "--strict",
]
if use_vmap:
    preflight_cmd.append("--require-gpu")
run_streaming(preflight_cmd, cwd=work_repo, label="full preflight", raise_on_error=True)

train_cmd = [
    "python", "-u", "train_progressive.py",
    "--mode", "complete",
    "--device", "auto",
    "--timesteps", "2000000",
    "--population", "8",
    "--generations", "10",
    "--output-dir", full_output_dir,
]
if use_vmap:
    train_cmd.append("--use-vmap")

run_streaming(train_cmd, cwd=work_repo, label="full training", raise_on_error=True)



## 7) Resume interrupted run (streaming)



In [ ]:
import os

work_repo = os.environ.get("ATOM_WORK_REPO", "/content/atom")
checkpoint_dir = os.environ.get("ATOM_RESUME_CHECKPOINT_DIR", "/content/drive/MyDrive/atom_runs/run1")
start_gen = os.environ.get("ATOM_RESUME_START_GEN", "8")
total_gens = os.environ.get("ATOM_RESUME_TOTAL_GENS", "20")
use_vmap = os.environ.get("ATOM_USE_VMAP", "1") == "1"

preflight_cmd = [
    "python", "-u", "-m", "src.atom.training.utils.colab_preflight",
    "--stage", "resume",
    "--checkpoint-dir", checkpoint_dir,
    "--strict",
]
if use_vmap:
    preflight_cmd.append("--require-gpu")
run_streaming(preflight_cmd, cwd=work_repo, label="resume preflight", raise_on_error=True)

resume_cmd = [
    "python", "-u", "scripts/training/resume_population_training.py",
    "--checkpoint-dir", checkpoint_dir,
    "--start-gen", str(start_gen),
    "--total-gens", str(total_gens),
]
if use_vmap:
    resume_cmd.append("--use-vmap")

run_streaming(resume_cmd, cwd=work_repo, label="resume run", raise_on_error=True)



In [ ]:
%%bash
set -euo pipefail
python -m pip install -U funkybob


## Troubleshooting

- Public repo: keep `ATOM_USE_GITHUB_TOKEN=0`.
- Private repo: set `ATOM_USE_GITHUB_TOKEN=1` and provide `GITHUB_TOKEN`.
- If Drive cache has local edits, bootstrap auto-stashes by default (`ATOM_DRIVE_REPO_SYNC_MODE=stash`).
- Use `ATOM_DRIVE_REPO_SYNC_MODE=reset` to discard cache edits, or `skip_pull` to avoid pulling when dirty.
- Use `python -m src.atom.training.utils.colab_preflight --stage <bootstrap|smoke|full|resume> --strict` for fast diagnostics.
- Milestone gate checklist: `docs/COLAB_VALIDATION_CHECKLIST.md`.

